# 04 — Cohortes y RFM

Las tres páginas anteriores miraban el negocio por el eje del **calendario**: cuántos suscriptores
hay en marzo, cuánto se factura en agosto. Ésta lo mira por el eje de la **edad**: qué le pasa a una
suscripción a los 3, 6 o 18 meses de haberse dado de alta, independientemente de en qué mes empezó.

Es el eje que hace falta para la página de cierre —CAC contra LTV por canal—, porque un canal barato
cuyas altas cancelan a los seis meses es caro, y eso sólo se ve en la curva de retención.

`docs/data_imperfections.md` cataloga cuatro trampas en este eje, y las cuatro se tratan aquí de
forma explícita:

1. **El descuento de bienvenida**, que genera un pico de cancelación previsible cuando se acaba y
   deja el ingreso del mes 0 sin comparar con el de las cohortes siguientes.
2. **Las suscripciones regaladas**, que no siguen el patrón de nadie: se cancelan cuando se acaba el
   regalo, no cuando el titular decide.
3. **Los downgrades de plan**, que hay que decidir explícitamente si cuentan como "sigue activo".
4. **El comprador de máquina con cápsulas de regalo**, que se suscribe semanas después y que un join
   ingenuo por fecha exacta pierde entero.

La segunda mitad de la página es un **RFM clásico** sobre la compra puntual, con su propia
imperfección: **más de un cuarto de los pedidos de tienda no tienen cliente** —seis de cada diez de
los de boutique— y no se pueden puntuar.

In [1]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "warehouse.duckdb").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

import utils_subscriptions as us

DB_PATH = PROJECT_ROOT / "data" / "warehouse.duckdb"
OUTPUT_PATH = PROJECT_ROOT / "analysis" / "outputs" / "cohorts_rfm.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- Parámetros del análisis ---
AS_OF = pd.Timestamp("2026-08-31")      # cierre del histórico
MIN_COHORT = 30                          # altas mínimas para que una cohorte entre en la curva
LTV_HORIZON = 36                         # meses a los que se proyecta el LTV
TAIL_MIN_COHORTS = 6                     # cohortes mínimas para fiarse de un punto de la curva
TAIL_WINDOW = 6                          # edades sobre las que se estima el hazard de la cola
RFM_QUANTILES = 5
ALPHA = 0.05

C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
C_VIOLET, C_RED = "#4a3aa7", "#e34948"
C_GRID, C_INK, C_MUTED = "#e6e6e3", "#0b0b0b", "#52514e"
CHANNEL_COLOR = {"paid_social": C_BLUE, "organic": C_AQUA, "influencer_code": C_ORANGE,
                 "referral": C_VIOLET, "podcast_ads": C_YELLOW, "direct_unknown": C_MUTED}
CHANNEL_LABEL = {"paid_social": "Paid social", "organic": "Orgánico",
                 "influencer_code": "Código influencer", "referral": "Referido",
                 "podcast_ads": "Podcast", "direct_unknown": "Directo / sin resolver"}
PLOT_LAYOUT = dict(
    template="plotly_white", height=420,
    margin=dict(l=70, r=30, t=60, b=50),
    font=dict(color=C_INK, size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor=C_GRID), yaxis=dict(gridcolor=C_GRID),
    hovermode="x unified",
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("proyecto:", PROJECT_ROOT.name, "| duckdb:", DB_PATH.exists())

proyecto: capsule-club-analytics | duckdb: True


## 1. Tres poblaciones que no se pueden mezclar

El panel mensual `fct_subscriptions_monthly` tiene una fila por suscripción y mes de vida, y
`dim_subscriptions` una por suscripción con sus atributos de alta. Lo primero es ver de qué está
hecha la base, porque **no es una población homogénea**: el descuento de bienvenida y el regalo
parten las altas en tres grupos disjuntos con comportamientos distintos.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)

facts = con.sql("select * from fct_subscriptions_monthly").df()
subs = con.sql("select * from dim_subscriptions").df()
customers = con.sql("select * from dim_customers").df()

for frame, cols in ((facts, ["month_start", "month_end", "cohort_month"]),
                    (subs, ["start_date", "cohort_month", "cancel_date"]),
                    (customers, ["first_signup_date", "signup_month", "first_subscription_date",
                                 "subscription_cohort_month", "as_of_date"])):
    for c in cols:
        frame[c] = pd.to_datetime(frame[c])

subs["grupo"] = np.where(subs.is_gifted, "regalada",
                 np.where(subs.had_welcome_discount, "con descuento", "sin descuento"))
facts["grupo"] = np.where(facts.is_gifted, "regalada",
                  np.where(facts.had_welcome_discount, "con descuento", "sin descuento"))

population = subs.groupby("grupo").agg(
    suscripciones=("subscription_id", "count"),
    canceladas=("status", lambda s: (s == "cancelled").sum()),
    vivas=("is_right_censored", "sum"),
    dias_medios=("observed_days", "mean"),
).assign(pct=lambda d: d.suscripciones / d.suscripciones.sum() * 100)
print(population.round(1).to_string())
print()
print(f"{len(subs):,} suscripciones · {facts.subscription_id.nunique():,} en el panel · "
      f"{len(facts):,} filas suscripción-mes")
print(f"cohortes de alta: {subs.cohort_month.min():%Y-%m} a {subs.cohort_month.max():%Y-%m} "
      f"({subs.cohort_month.nunique()} meses)")
print(f"censura por la derecha: {subs.is_right_censored.sum():,} suscripciones siguen vivas al "
      f"cierre ({subs.is_right_censored.mean():.1%})")

               suscripciones  canceladas  vivas  dias_medios   pct
grupo                                                             
con descuento           2680        1292   1388        263.3  52.4
regalada                 327         218    109        197.3   6.4
sin descuento           2103         743   1360        335.5  41.2

5,110 suscripciones · 5,110 en el panel · 52,171 filas suscripción-mes
cohortes de alta: 2023-09 a 2026-08 (36 meses)
censura por la derecha: 2,857 suscripciones siguen vivas al cierre (55.9%)


Los tres grupos son **disjuntos por construcción**: ninguna suscripción regalada lleva descuento de
bienvenida. Eso simplifica el análisis —no hay que desentrelazar dos efectos sobre la misma
suscripción— pero obliga a decidir qué se hace con cada grupo antes de dibujar una sola curva.

La censura por la derecha es lo primero que condiciona todo lo demás: **el 56% de las suscripciones
sigue viva al cierre del histórico**, así que cualquier métrica que necesite "cuánto duró" está
sistemáticamente sesgada a la baja si se calcula sólo sobre las que ya cancelaron. Por eso todo lo
que sigue se construye como curva de retención por edad —que usa la información de las vivas hasta
donde llega— y no como media de duración.

## 2. Qué cuenta como "seguir activo"

`dbt_project/README.md` avisa de que el mart ofrece tres definiciones de activo. Para una curva de
retención la elección no es cosmética, y la que parece más natural es la equivocada.

- `is_active_eom`: la suscripción no está cancelada a fin de mes. **Ignora las pausas.**
- `is_active_net_of_pauses`: además exige que no esté pausada ese mes.

Las pausas estacionales de verano y Navidad son una imperfección documentada: *no son cancelación*.
Si la retención se mide sobre los activos netos, la pausa de agosto entra en la curva como "churn a
los N meses" —y N es distinto para cada cohorte, porque cada una llegó a agosto con una edad
distinta—. El resultado es una curva que mezcla edad con calendario.

In [3]:
gifted_mask = facts.is_gifted
base = facts[~gifted_mask]

curve_eom = us.build_retention_curve(base, min_cohort=MIN_COHORT)
net = base.copy()
net["is_active_eom"] = net["is_active_net_of_pauses"]
curve_net = us.build_retention_curve(net, min_cohort=MIN_COHORT)

compare = pd.DataFrame({"is_active_eom": curve_eom.values,
                        "is_active_net_of_pauses": curve_net.values}).head(19) * 100
compare["dif_pp"] = compare.is_active_eom - compare.is_active_net_of_pauses
print("Retención por edad, según la definición de activo (%):")
print(compare.round(1).to_string())

# La pausa es un fenómeno de calendario: se concentra en meses naturales concretos.
pause_by_month = (facts.groupby(facts.month_start.dt.month).is_paused.mean() * 100)
MONTHS = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("Retención por edad de la suscripción",
                                    "% de suscripciones en pausa, por mes natural"))
for name, curve, color in (("is_active_eom", curve_eom, C_BLUE),
                           ("is_active_net_of_pauses", curve_net, C_ORANGE)):
    fig.add_trace(go.Scatter(x=curve.values.index, y=curve.values.to_numpy() * 100,
                             name=name, mode="lines+markers",
                             line=dict(color=color, width=2.5)), row=1, col=1)
fig.add_trace(go.Bar(x=MONTHS, y=pause_by_month.to_numpy(), marker_color=C_AQUA,
                     showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="meses desde el alta", row=1, col=1)
fig.update_yaxes(title_text="% retenido", row=1, col=1)
fig.update_yaxes(title_text="% en pausa", row=1, col=2)
fig.update_layout(**{**PLOT_LAYOUT, "height": 400, "hovermode": "closest"},
                  title="La pausa es calendario, no edad")
fig.show()

peak = pause_by_month.idxmax()
print()
print(f"la pausa hace techo en {MONTHS[peak - 1]} ({pause_by_month.max():.1f}%) y suelo en "
      f"{MONTHS[pause_by_month.idxmin() - 1]} ({pause_by_month.min():.1f}%)")

Retención por edad, según la definición de activo (%):
                    is_active_eom  is_active_net_of_pauses  dif_pp
months_since_start                                                
0                           100.0                    100.0     0.0
1                            93.9                     92.0     1.8
2                            83.9                     81.2     2.7
3                            81.0                     77.6     3.4
4                            78.6                     75.2     3.4
5                            76.0                     72.6     3.4
6                            73.3                     69.3     4.0
7                            71.2                     67.7     3.5
8                            68.9                     65.8     3.1
9                            66.8                     63.1     3.7
10                           64.3                     61.5     2.8
11                           61.3                     58.6     2.7
12     


la pausa hace techo en Ago (13.5%) y suelo en May (0.0%)


La diferencia es de **2 a 4 puntos en todas las edades** y no se cierra nunca, porque cada mes que
pasa hay una fracción de la base en pausa que la definición neta cuenta como perdida y que vuelve.
El panel de la derecha lo confirma: la pausa es un fenómeno de **mes natural** —13,5% de la base en
agosto y literalmente nadie en mayo— no de edad de la suscripción.

**Decisión: la retención se mide sobre `is_active_eom`.** Es la misma que usa
`analysis/utils_subscriptions.py` para el modelo de cohortes de las páginas 2 y 3, así que las tres
páginas hablan de lo mismo. La pausa se trata donde corresponde, como factor de calendario, y ahí ya
la trató la página de suscriptores.

## 3. Las suscripciones regaladas no son una cohorte

327 suscripciones son un regalo. El catálogo de imperfecciones avisa de que "no siguen el patrón
normal de reactivación/cancelación del titular real del pago", y conviene ver exactamente en qué se
traduce eso antes de decidir qué hacer con ellas.

In [4]:
gift_curve = us.build_retention_curve(facts[gifted_mask], min_cohort=5)
gift_compare = pd.DataFrame({"regaladas": gift_curve.values,
                             "no regaladas": curve_eom.values}).head(15) * 100
print("Retención (%):")
print(gift_compare.round(1).to_string())
print()

gift_hazard = pd.DataFrame({"regaladas": gift_curve.hazard,
                            "no regaladas": curve_eom.hazard}).head(10) * 100
print("Hazard por edad (% de los vivos que cancelan ese mes):")
print(gift_hazard.round(1).to_string())
print()

# El motivo de cancelación no distingue a las regaladas: se recicla del mismo catálogo.
reasons = (subs[subs.status == "cancelled"]
           .pivot_table(index="cancel_reason", columns="is_gifted",
                        values="subscription_id", aggfunc="count", fill_value=0))
reasons.columns = ["no regalada", "regalada"]
reasons["% regaladas"] = reasons["regalada"] / reasons.sum(axis=1).where(lambda s: s > 0) * 100
print("Motivo declarado de cancelación:")
print(reasons.round(1).to_string())

fig = go.Figure()
for name, curve, color in (("Regaladas", gift_curve, C_ORANGE),
                           ("No regaladas", curve_eom, C_BLUE)):
    fig.add_trace(go.Scatter(x=curve.values.index[:19], y=curve.values.to_numpy()[:19] * 100,
                             name=name, mode="lines+markers", line=dict(color=color, width=2.5)))
fig.update_layout(**{**PLOT_LAYOUT, "height": 400},
                  title="La suscripción regalada cae en escalón, no en curva",
                  xaxis_title="meses desde el alta", yaxis_title="% retenido")
fig.show()

alive_gift = gift_curve.values
cliff = (alive_gift.loc[3] - alive_gift.loc[6]) * 100
print()
print(f"caída de las regaladas entre el mes 3 y el mes 6: {cliff:.0f} puntos "
      f"({alive_gift.loc[3]:.1%} -> {alive_gift.loc[6]:.1%})")

Retención (%):
                    regaladas  no regaladas
months_since_start                         
0                       100.0         100.0
1                        99.3          93.9
2                        98.0          83.9
3                        87.7          81.0
4                        62.7          78.6
5                        39.5          76.0
6                        23.0          73.3
7                        25.4          71.2
8                        26.3          68.9
9                        27.8          66.8
10                       26.9          64.3
11                       24.2          61.3
12                       24.4          59.4
13                       24.6          57.6
14                       24.6          55.6

Hazard por edad (% de los vivos que cancelan ese mes):
                    regaladas  no regaladas
months_since_start                         
1                         0.7           6.1
2                         1.4          10.7
3    


caída de las regaladas entre el mes 3 y el mes 6: 65 puntos (87.7% -> 23.0%)


**La curva de las regaladas no es una curva, es un acantilado.** Se mantienen por encima del 87%
hasta el mes 3 —mejor que las normales, porque nadie cancela un regalo que no paga— y entre el mes 3
y el 6 pierden 65 puntos de golpe. Es el final del término del regalo, no una decisión de consumo.

Y hay una trampa dentro de la trampa: **el motivo de cancelación no las delata**. No existe ningún
código de "fin de regalo": se reparten entre los mismos cinco motivos que el resto —precio, sabor,
mudanza— porque el motivo se recicla del mismo catálogo. Quien intente aislar el churn de regalo
filtrando por `cancel_reason` no encontrará nada; hay que mirar `is_gifted` y la forma de la curva.

Conviene fijarse también en que **el hazard de las regaladas sale negativo en los meses 7 a 9**. No
es un error de cálculo: la curva agregada divide vivos entre altas de las cohortes que han llegado a
esa edad, y a partir del mes 7 sólo llegan las cohortes de regalo más antiguas, que son las de
término más largo. La composición cambia y la curva sube. Es la tercera señal de que a esta
población no se le puede aplicar la misma herramienta que al resto.

**Decisión: las regaladas se excluyen de la curva principal y se reportan aparte.** Mezclarlas mete
un acantilado artificial en el mes 4-6 de la curva agregada, y ese tramo es justo el que usa la
página 2 para proyectar activos. No se tiran: el regalo es un canal de captación real y su pregunta
de negocio —cuántos se quedan después del término— se responde con su propia curva, que se estabiliza
en torno al 25%.

## 4. El descuento de bienvenida

El 56% de las altas no regaladas llevan descuento el primer mes. El catálogo de imperfecciones dice
que eso "genera un pico de cancelación previsible en el mes 2 cuando se empieza a cobrar el precio
completo" y que "la cohorte de mes 0 no es comparable en ingreso con las siguientes".

Antes de restar dos curvas conviene comprobar una cosa que casi nunca se comprueba: **¿a quién se le
dio el descuento?** Si se hubiera dado a los clientes con peor pinta, la diferencia de retención
mediría la selección y no el descuento.

In [5]:
ng_subs = subs[~subs.is_gifted].copy()
balance_rows = []
for dim in ("acquisition_channel", "tier", "current_plan"):
    table = pd.crosstab(ng_subs[dim], ng_subs.had_welcome_discount)
    chi2, p, dof, _ = chi2_contingency(table)
    share = (table[True] / table.sum(axis=1) * 100)
    balance_rows.append({"variable": dim, "categorias": len(table),
                         "pct_min": share.min(), "pct_max": share.max(),
                         "chi2": chi2, "p_valor": p})
year = ng_subs.start_date.dt.year
table = pd.crosstab(year, ng_subs.had_welcome_discount)
chi2, p, dof, _ = chi2_contingency(table)
share = (table[True] / table.sum(axis=1) * 100)
balance_rows.append({"variable": "año de alta", "categorias": len(table),
                     "pct_min": share.min(), "pct_max": share.max(),
                     "chi2": chi2, "p_valor": p})
balance = pd.DataFrame(balance_rows)
print("¿Está el descuento repartido al azar? (% con descuento dentro de cada categoría)")
print(balance.round(3).to_string(index=False))
print()
print(f"global: {ng_subs.had_welcome_discount.mean():.1%} de las altas no regaladas")

¿Está el descuento repartido al azar? (% con descuento dentro de cada categoría)
           variable  categorias  pct_min  pct_max  chi2  p_valor
acquisition_channel           6   53.106   57.768 3.577    0.612
               tier           4   55.421   59.167 2.794    0.425
       current_plan           3   55.008   56.557 0.885    0.642
        año de alta           4   50.711   56.989 3.272    0.352

global: 56.0% de las altas no regaladas


**El descuento está repartido al azar.** Dentro de cada canal, tier, plan y año de alta la proporción
de descuentos se mueve en una banda estrecha alrededor del 56% global, y ningún contraste χ² rechaza
la independencia al 5%.

Eso es lo que autoriza a leer la diferencia entre las dos curvas como el **efecto** del descuento y
no como una diferencia entre dos tipos de cliente. En un dataset real esta celda sería el paso
obligatorio antes de la resta; aquí sale limpia, pero la conclusión sólo vale porque se ha mirado.

In [6]:
disc = us.build_retention_curve(base[base.had_welcome_discount], min_cohort=MIN_COHORT)
plain = us.build_retention_curve(base[~base.had_welcome_discount], min_cohort=MIN_COHORT)

retention_by_discount = pd.DataFrame({
    "con_descuento": disc.values, "sin_descuento": plain.values}).head(19) * 100
retention_by_discount["dif_pp"] = (retention_by_discount.con_descuento
                                   - retention_by_discount.sin_descuento)
hazard_by_discount = pd.DataFrame({
    "h_con_descuento": disc.hazard, "h_sin_descuento": plain.hazard}).head(10) * 100
hazard_by_discount["exceso_pp"] = (hazard_by_discount.h_con_descuento
                                   - hazard_by_discount.h_sin_descuento)
print("Retención (%):")
print(retention_by_discount.round(1).to_string())
print()
print("Hazard por edad (%):")
print(hazard_by_discount.round(2).to_string())

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=("Retención por edad", "Hazard: % de vivos que cancelan ese mes"))
for name, curve, color in (("Con descuento", disc, C_ORANGE), ("Sin descuento", plain, C_BLUE)):
    fig.add_trace(go.Scatter(x=curve.values.index[:19], y=curve.values.to_numpy()[:19] * 100,
                             name=name, mode="lines+markers", line=dict(color=color, width=2.5)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=curve.hazard.index[:13], y=curve.hazard.to_numpy()[:13] * 100,
                             name=name, mode="lines", line=dict(color=color, width=2.5),
                             showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="meses desde el alta")
fig.update_yaxes(title_text="% retenido", row=1, col=1)
fig.update_yaxes(title_text="% que cancela", row=1, col=2)
fig.update_layout(**{**PLOT_LAYOUT, "height": 400, "hovermode": "closest"},
                  title="El descuento de bienvenida y su factura, en los meses 1 y 2")
fig.show()

Retención (%):
                    con_descuento  sin_descuento  dif_pp
months_since_start                                      
0                           100.0          100.0     0.0
1                            90.7           97.7    -7.1
2                            75.1           94.8   -19.8
3                            72.1           92.3   -20.2
4                            69.6           89.8   -20.1
5                            67.3           86.7   -19.4
6                            65.2           83.3   -18.1
7                            63.2           81.1   -17.8
8                            60.4           79.2   -18.7
9                            57.9           77.9   -20.0
10                           56.0           74.4   -18.4
11                           53.1           71.1   -18.0
12                           51.4           68.9   -17.5
13                           49.2           67.5   -18.3
14                           47.6           64.9   -17.2
15              

In [7]:
# Exceso de cancelaciones atribuible al descuento: la diferencia de hazard aplicada a los expuestos.
exposure = base.pivot_table(index="months_since_start", columns="had_welcome_discount",
                            values="is_active_eom", aggfunc="sum")
churns = base.pivot_table(index="months_since_start", columns="had_welcome_discount",
                          values="is_churn_month", aggfunc="sum")
hazard_raw = churns / exposure
excess = ((hazard_raw[True] - hazard_raw[False]) * exposure[True]).round(0)
spike_ages = [1, 2]
print("Exceso de cancelaciones por edad (cancelaciones observadas menos las esperadas "
      "al hazard del grupo sin descuento):")
print(excess.head(8).to_string())
print()
print(f"total en los meses {spike_ages}: {excess.loc[spike_ages].sum():.0f} cancelaciones "
      f"({excess.loc[spike_ages].sum() / exposure[True].loc[0] * 100:.1f}% de las altas con descuento)")
print(f"resto de edades (3 a {int(excess.index.max())}): {excess.loc[3:].sum():.0f}")

# Ingreso del mes 0: el descuento no se ve en el MRR contratado, sólo en el reconocido.
revenue_age = (base[base.is_active_eom]
               .pivot_table(index="months_since_start", columns="had_welcome_discount",
                            values=["contracted_mrr_eur", "recognized_revenue_eur"],
                            aggfunc="mean"))
revenue_age.columns = ["mrr_sin", "mrr_con", "reconocido_sin", "reconocido_con"]
revenue_age = revenue_age[["mrr_con", "mrr_sin", "reconocido_con", "reconocido_sin"]].head(6)
revenue_age["gap_reconocido_pct"] = (revenue_age.reconocido_con / revenue_age.reconocido_sin - 1) * 100
print()
print("Ingreso medio por suscripción activa, según edad (€):")
print(revenue_age.round(2).to_string())

Exceso de cancelaciones por edad (cancelaciones observadas menos las esperadas al hazard del grupo sin descuento):
months_since_start
0      0.0
1    184.0
2    335.0
3     19.0
4     13.0
5      3.0
6     -5.0
7      8.0

total en los meses [1, 2]: 519 cancelaciones (19.4% de las altas con descuento)
resto de edades (3 a 35): 55



Ingreso medio por suscripción activa, según edad (€):
                    mrr_con  mrr_sin  reconocido_con  reconocido_sin  gap_reconocido_pct
months_since_start                                                                      
0                     29.87    29.97            8.33           16.96              -50.88
1                     29.35    29.34           19.36           28.92              -33.05
2                     28.85    29.15           23.15           29.32              -21.04
3                     28.61    28.77           24.03           28.43              -15.45
4                     28.57    28.70           26.28           27.56               -4.63
5                     28.43    28.69           26.34           27.49               -4.18


**El efecto está donde el catálogo dice y se apaga donde debe.** El hazard del grupo con descuento
es 7,1 puntos más alto en el mes 1 y **14,2 puntos más alto en el mes 2**; a partir del mes 3 el
exceso se desploma por debajo de 1,3 puntos y cambia de signo varias veces, que es lo que se espera
de dos grupos que ya se comportan igual. La separación entre las dos curvas hace techo en 20 puntos
hacia el mes 3, se mantiene ahí hasta el mes 9 y después **se estrecha** —12 puntos al mes 18—
porque el grupo sin descuento también va cayendo y el que sobrevivió a la criba ya no.

Eso cambia la lectura de negocio por completo. El descuento no deteriora la retención a largo plazo:
**produce una criba única al acabarse**, de unas 520 cancelaciones —el 19% de las altas con
descuento— concentradas en dos meses. Quien se queda después del mes 3 se comporta exactamente igual
que quien nunca tuvo descuento. La pregunta correcta no es "¿el descuento daña la retención?" sino
"¿el 81% que sobrevive compensa el coste del descuento y de captar a los que se van?", y ésa es la
pregunta de la página 7.

**Y el ingreso del mes 0 no es comparable, pero sólo en una de las tres medidas.** El MRR contratado
es idéntico en los dos grupos desde el primer mes (29,87 € contra 29,97 €): el descuento no cambia la
tarifa vigente. Lo que cambia es el **ingreso reconocido**, que en el mes 0 es la mitad —8,33 €
contra 16,96 €—. Quien compare cohortes por MRR no verá el descuento; quien las compare por ingreso
reconocido verá un mes 0 hundido que no es un problema de captación.

## 5. Downgrades: la decisión que el catálogo deja abierta

`docs/data_imperfections.md` plantea los downgrades como una pregunta explícita: *"reducen el valor
sin ser churn — hay que decidir explícitamente cómo tratarlos en la curva de retención (¿cuenta como
'sigue activo' pese a generar menos ingreso?)"*.

Para responderla hay que saber primero qué es un downgrade en estos datos, y la respuesta no es la
que sugiere la pregunta.

In [8]:
panel = facts.sort_values(["subscription_id", "months_since_start"]).copy()
grouped = panel.groupby("subscription_id", sort=False)
panel["prev_plan"] = grouped.plan_at_month.shift(1)
panel["prev_mrr"] = grouped.contracted_mrr_eur.shift(1)

PLAN_MONTHS = {"monthly": 1, "quarterly": 3, "annual": 12}
transitions = []
for flag, label in (("is_downgrade_month", "downgrade"), ("is_upgrade_month", "upgrade")):
    moves = panel[panel[flag] & panel.prev_plan.notna()]
    valid = moves[(moves.prev_mrr > 0) & (moves.contracted_mrr_eur > 0)]
    for (old, new), n in moves.groupby(["prev_plan", "plan_at_month"]).size().items():
        transitions.append({"tipo": label, "de": old, "a": new, "n": int(n)})
    print(f"--- {label}: {len(moves)} cambios en "
          f"{moves.subscription_id.nunique()} suscripciones")
    for (old, new), n in moves.groupby(["prev_plan", "plan_at_month"]).size().sort_values(
            ascending=False).items():
        print(f"      {old:>9s} -> {new:<9s} {n:4d}")
    print(f"      MRR contratado: {valid.prev_mrr.mean():.2f} € -> "
          f"{valid.contracted_mrr_eur.mean():.2f} € "
          f"({valid.contracted_mrr_eur.mean() / valid.prev_mrr.mean() - 1:+.1%})")
    print(f"      meses de compromiso: {moves.prev_plan.map(PLAN_MONTHS).mean():.1f} -> "
          f"{moves.plan_at_month.map(PLAN_MONTHS).mean():.1f}")
    print()

print("MRR mensual medio por plan (suscripciones activas):")
print(facts[facts.is_active_eom].groupby("plan_at_month").contracted_mrr_eur.mean()
      .round(2).to_string())

--- downgrade: 208 cambios en 200 suscripciones
      quarterly -> monthly    122
         annual -> quarterly   86
      MRR contratado: 28.27 € -> 30.05 € (+6.3%)
      meses de compromiso: 6.7 -> 1.8

--- upgrade: 322 cambios en 307 suscripciones
        monthly -> quarterly  228
      quarterly -> annual      93
        monthly -> annual       1
      MRR contratado: 30.22 € -> 28.51 € (-5.7%)
      meses de compromiso: 1.6 -> 5.6

MRR mensual medio por plan (suscripciones activas):


plan_at_month
annual       25.99
monthly      29.79
quarterly    28.02


In [9]:
# ¿El downgrade adelanta la cancelación? Se compara el hazard posterior al cambio con el de
# quienes nunca lo hicieron, a partir de la misma edad, para no comparar meses jóvenes con viejos.
ng_panel = panel[~panel.is_gifted].copy()
first_down = ng_panel[ng_panel.is_downgrade_month].groupby("subscription_id").months_since_start.min()
ng_panel["downgrade_age"] = ng_panel.subscription_id.map(first_down)
after_downgrade = ng_panel[ng_panel.downgrade_age.notna()
                           & (ng_panel.months_since_start > ng_panel.downgrade_age)]
never_downgraded = ng_panel[~ng_panel.subscription_id.isin(first_down.index)]
median_age = int(first_down.median())
control = never_downgraded[never_downgraded.months_since_start > median_age]

h_after = after_downgrade.is_churn_month.sum() / len(after_downgrade)
h_control = control.is_churn_month.sum() / len(control)
print(f"edad mediana del downgrade: mes {median_age}")
print(f"hazard tras el downgrade        : {h_after:.2%}/mes "
      f"({int(after_downgrade.is_churn_month.sum())} de {len(after_downgrade)} meses-riesgo)")
print(f"hazard control (nunca, edad > {median_age}): {h_control:.2%}/mes "
      f"({int(control.is_churn_month.sum())} de {len(control)} meses-riesgo)")
print(f"diferencia: {(h_after - h_control) * 100:+.2f} puntos")

edad mediana del downgrade: mes 8
hazard tras el downgrade        : 3.83%/mes (61 de 1592 meses-riesgo)
hazard control (nunca, edad > 8): 3.89%/mes (662 de 17003 meses-riesgo)
diferencia: -0.06 puntos


**Un downgrade en estos datos no baja el precio: acorta el compromiso.** Las transiciones marcadas
son `annual → quarterly` y `quarterly → monthly`, es decir, pasar a un ciclo más corto. Y como el
ciclo largo lleva descuento por compromiso (anual 25,99 €/mes, trimestral 28,02 €, mensual 29,79 €),
acortarlo **sube el MRR contratado un 6,3%** y baja el compromiso de 6,7 a 1,8 meses. El upgrade hace
lo contrario: baja el MRR un 5,7% y alarga el compromiso.

Tampoco adelanta la cancelación: el hazard después del downgrade es del 3,83% mensual frente al 3,89%
de quienes nunca lo hicieron a la misma edad. **Seis centésimas de punto, en la dirección contraria a
la esperada.**

**Decisión: los downgrades cuentan como "sigue activo" en la curva de retención, sin asterisco.** Con
estos datos la premisa de la pregunta —"generan menos ingreso"— es falsa en MRR, y el riesgo real que
introducen no es de tarifa sino de horizonte: un cliente mensual puede irse el mes que viene y uno
anual no. Eso se mide con el compromiso, no con la curva.

> **Para revisar con Diego.** `docs/data_imperfections.md` describe el downgrade como algo que
> "reduce el valor", y el comentario del generador dice lo mismo (`downgrade = menos ingreso`), pero
> el código mueve el plan hacia el ciclo **corto**, que es el más caro por mes. O la intención era
> mover el *tier* (classic/intense/decaf/explorer), que sí tiene precios distintos, o el catálogo
> debería decir que lo que se reduce es el compromiso. No lo he cambiado: no es una anomalía de los
> datos sino una discrepancia entre la documentación y el generador, y la decisión es de diseño.

In [10]:
# Tres curvas, no dos. El mart lleva la tarifa a `paused_mrr_eur` y deja `contracted_mrr_eur`
# a cero mientras la suscripción está en pausa, así que "retención de ingreso" a secas mezcla
# dos cosas distintas: cuánta tarifa sobrevive y cuánta de esa tarifa se está cobrando.
#
#   logo          -> qué fracción de las altas sigue sin cancelar
#   mrr_tarifa    -> qué fracción del MRR inicial sigue contratada, pausas incluidas
#   mrr_cobrable  -> qué fracción se está cobrando de verdad este mes
#
# logo - mrr_tarifa aísla el efecto de MIX (si sobreviven los planes caros o los baratos).
# mrr_tarifa - mrr_cobrable aísla el efecto de PAUSA.
base = base.copy()
base["gross_mrr_eur"] = base.contracted_mrr_eur + base.paused_mrr_eur

cohort_size = base[base.months_since_start == 0].groupby("cohort_month").is_active_eom.sum()
eligible = cohort_size[cohort_size >= MIN_COHORT].index
mrr_start = (base[(base.months_since_start == 0) & base.is_active_eom]
             .groupby("cohort_month").gross_mrr_eur.sum())

def cohort_ratio(column):
    pivot = (base[base.is_active_eom]
             .pivot_table(index="cohort_month", columns="months_since_start",
                          values=column, aggfunc="sum")
             .loc[eligible])
    observed = pivot.notna()
    return (pivot.sum(axis=0, skipna=True)
            / observed.mul(mrr_start.reindex(pivot.index), axis=0).sum(axis=0))

logo_vs_revenue = pd.DataFrame({
    "logo": curve_eom.values,
    "mrr_tarifa": cohort_ratio("gross_mrr_eur"),
    "mrr_cobrable": cohort_ratio("contracted_mrr_eur")}).head(19) * 100
logo_vs_revenue["efecto_mix_pp"] = logo_vs_revenue.logo - logo_vs_revenue.mrr_tarifa
logo_vs_revenue["efecto_pausa_pp"] = logo_vs_revenue.mrr_tarifa - logo_vs_revenue.mrr_cobrable
logo_vs_revenue["gap_total_pp"] = logo_vs_revenue.logo - logo_vs_revenue.mrr_cobrable
print("Retención de logo, de tarifa y de ingreso cobrable (%):")
print(logo_vs_revenue.round(2).to_string())

mature_ages = slice(3, 18)
effects = logo_vs_revenue.loc[mature_ages,
                              ["efecto_mix_pp", "efecto_pausa_pp", "gap_total_pp"]].mean()
pause_share = effects.efecto_pausa_pp / effects.gap_total_pp * 100
print()
print("Medias sobre las edades 3-18 (puntos):")
print(effects.round(2).to_string())
print()
print(f"la pausa explica el {pause_share:.0f}% del hueco; el mix de plan, el resto")
paused_total = float(base.paused_mrr_eur.sum())
gross_total = float(base.gross_mrr_eur.sum())
print(f"MRR retirado por pausas: {paused_total:,.0f} € · "
      f"{paused_total / gross_total:.2%} del MRR de tarifa · "
      f"{int(base[base.is_paused].subscription_id.nunique()):,} suscripciones")

plan_mix = (base[base.is_active_eom]
            .pivot_table(index="months_since_start", columns="plan_at_month",
                         values="subscription_id", aggfunc="count"))
plan_mix = (plan_mix.div(plan_mix.sum(axis=1), axis=0) * 100)
print()
print("Mix de plan entre las suscripciones vivas, por edad (%):")
print(plan_mix.head(19).round(1).to_string())

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=("Logo, tarifa e ingreso cobrable",
                                    "Mix de plan entre los vivos (%)"))
for col, name, color in (("logo", "Logo", C_BLUE),
                         ("mrr_tarifa", "MRR de tarifa", C_AQUA),
                         ("mrr_cobrable", "MRR cobrable", C_ORANGE)):
    fig.add_trace(go.Scatter(x=logo_vs_revenue.index, y=logo_vs_revenue[col], name=name,
                             mode="lines", line=dict(color=color, width=2.5)), row=1, col=1)
for plan, color in (("monthly", C_BLUE), ("quarterly", C_AQUA), ("annual", C_VIOLET)):
    fig.add_trace(go.Scatter(x=plan_mix.index[:19], y=plan_mix[plan].to_numpy()[:19], name=plan,
                             mode="lines", stackgroup="one", line=dict(width=0.5, color=color)),
                  row=1, col=2)
fig.update_xaxes(title_text="meses desde el alta")
fig.update_layout(**{**PLOT_LAYOUT, "height": 400, "hovermode": "closest"},
                  title="El hueco entre logo e ingreso lo abre la pausa, no el plan")
fig.show()

Retención de logo, de tarifa y de ingreso cobrable (%):
                      logo  mrr_tarifa  mrr_cobrable  efecto_mix_pp  efecto_pausa_pp  gap_total_pp
months_since_start                                                                                
0                   100.00      100.00        100.00           0.00             0.00          0.00
1                    93.87       93.93         92.07          -0.05             1.85          1.80
2                    83.87       83.91         81.26          -0.04             2.65          2.61
3                    81.04       81.07         77.67          -0.03             3.40          3.37
4                    78.57       78.60         75.22          -0.04             3.39          3.35
5                    75.99       75.99         72.55           0.00             3.43          3.43
6                    73.29       73.27         69.20           0.02             4.08          4.10
7                    71.21       71.15         67.62 


Medias sobre las edades 3-18 (puntos):
efecto_mix_pp     -0.06
efecto_pausa_pp    2.96
gap_total_pp       2.90

la pausa explica el 102% del hueco; el mix de plan, el resto
MRR retirado por pausas: 55,360 € · 3.87% del MRR de tarifa · 1,064 suscripciones

Mix de plan entre las suscripciones vivas, por edad (%):
plan_at_month       annual  monthly  quarterly
months_since_start                            
0                     13.6     61.8       24.6
1                     13.3     61.8       24.9
2                     13.4     61.0       25.6
3                     13.6     60.5       25.9
4                     13.7     60.0       26.3
5                     14.0     59.3       26.8
6                     14.2     58.8       27.0
7                     14.2     58.7       27.1
8                     13.8     58.7       27.5
9                     13.8     57.8       28.5
10                    14.0     57.4       28.6
11                    13.9     57.2       29.0
12                    13.6  

La retención de ingreso va entre **2 y 4 puntos por debajo** de la de logo en todas las edades, y la
descomposición dice exactamente de dónde salen esos puntos.

**No de los downgrades**, que ya sabíamos que suben el MRR. **Y tampoco del mix de plan**, que es la
explicación que pide a gritos el panel de la derecha: el plan mensual pasa del 61,8% de los vivos en
el mes 0 al 56,0% en el mes 14, empujado por los upgrades (322 cambios frente a 208 downgrades) y por
la mayor supervivencia de quien se compromete a largo. El mix **se mueve de verdad**… y no mueve el
dinero: su efecto medio entre los meses 3 y 18 es de **−0,06 puntos**. Las diferencias de tarifa
entre ciclos (26 € a 30 €) son demasiado pequeñas para que un desplazamiento de cinco puntos del mix
se note en euros.

**El hueco entero lo abre la pausa**: 2,96 puntos de media, el 102% del total. Y tiene todo el
sentido, porque es el único evento del catálogo que **retira ingreso sin retirar cliente** — durante
la pausa el mart lleva la tarifa a `paused_mrr_eur` y deja `contracted_mrr_eur` a cero—. Son 55.360 €
de MRR retirado, el 3,9% del de tarifa, sobre 1.064 suscripciones.

Aquí está, por tanto, la respuesta a la pregunta que el catálogo dejaba abierta —*¿cuenta como
"sigue activo" algo que genera menos ingreso?*—, sólo que el protagonista no es el que anunciaba:

- **En la curva de logo, la pausa cuenta como activa.** No es una decisión de abandono y volverá.
- **En la de ingreso, no puede contar**, porque ese mes no entra un euro.
- Y las dos curvas hay que publicarlas juntas, porque la distancia entre ellas **es** la métrica:
  cuánto ingreso está aparcado en clientes que no se han ido.

La lección de método es la que más vale: la explicación intuitiva (el mix) tenía a su favor un
gráfico que la confirmaba, y era falsa. Un gráfico que enseña que algo *se mueve* no demuestra que
ese algo *importe*; para eso hay que descomponer y medir su contribución.

## 6. La matriz de cohortes

Hasta aquí todo eran curvas agregadas sobre el eje de la edad. La matriz cohorte × edad enseña lo que
el agregado promedia: si las cohortes recientes retienen mejor o peor que las antiguas.

In [11]:
cohort_pivot = base.pivot_table(index="cohort_month", columns="months_since_start",
                                values="is_active_eom", aggfunc="sum")
cohort_sizes = cohort_pivot[0]
cohort_pivot = cohort_pivot[cohort_sizes >= MIN_COHORT]
cohort_sizes = cohort_sizes[cohort_sizes >= MIN_COHORT]
cohort_matrix = cohort_pivot.div(cohort_sizes, axis=0) * 100

print(f"{len(cohort_matrix)} cohortes con al menos {MIN_COHORT} altas "
      f"(de {base.cohort_month.nunique()} meses)")
print()
print("Retención por cohorte y edad (%), una fila por mes de alta:")
print(cohort_matrix.iloc[:, :19].round(0).to_string())

fig = go.Figure(go.Heatmap(
    z=cohort_matrix.iloc[:, :25].to_numpy(),
    x=list(cohort_matrix.columns[:25]),
    y=[d.strftime("%Y-%m") for d in cohort_matrix.index],
    colorscale=[[0, "#f4f4f2"], [0.5, "#7fb3e8"], [1, "#2a78d6"]],
    colorbar=dict(title="% vivo"), hovertemplate="cohorte %{y} · mes %{x}: %{z:.0f}%<extra></extra>"))
fig.update_layout(**{**PLOT_LAYOUT, "height": 620, "hovermode": "closest"},
                  title="Matriz de cohortes: % de la cohorte que sigue activo",
                  xaxis_title="meses desde el alta", yaxis_title="cohorte de alta")
fig.show()

by_year = cohort_matrix.assign(anio=cohort_matrix.index.year).groupby("anio")[[3, 6, 12]].mean()
by_year["cohortes"] = cohort_matrix.assign(anio=cohort_matrix.index.year).groupby("anio").size()
print()
print("Retención media por año de cohorte (%):")
print(by_year.round(1).to_string())
at_6 = cohort_matrix[6].dropna()
print()
print(f"mejor cohorte al mes 6: {at_6.idxmax():%Y-%m} ({at_6.max():.1f}%) · "
      f"peor: {at_6.idxmin():%Y-%m} ({at_6.min():.1f}%)")

36 cohortes con al menos 30 altas (de 36 meses)

Retención por cohorte y edad (%), una fila por mes de alta:
months_since_start     0     1     2     3     4     5     6     7     8     9     10    11    12    13    14    15    16    17    18
cohort_month                                                                                                                         
2023-09-01          100.0  97.0  84.0  81.0  75.0  69.0  59.0  59.0  59.0  59.0  56.0  50.0  47.0  47.0  47.0  44.0  44.0  44.0  44.0
2023-10-01          100.0  96.0  84.0  82.0  82.0  80.0  80.0  76.0  75.0  71.0  69.0  67.0  67.0  64.0  62.0  62.0  62.0  60.0  56.0
2023-11-01          100.0  97.0  92.0  92.0  90.0  87.0  84.0  84.0  84.0  77.0  77.0  75.0  75.0  72.0  72.0  69.0  66.0  62.0  62.0
2023-12-01          100.0  95.0  89.0  81.0  79.0  77.0  74.0  73.0  68.0  66.0  65.0  65.0  61.0  60.0  56.0  52.0  52.0  50.0  50.0
2024-01-01          100.0  98.0  87.0  87.0  85.0  82.0  79.0  79.0  78.0  78.0  77.0  


Retención media por año de cohorte (%):
months_since_start     3     6    12  cohortes
anio                                          
2023                83.9  74.3  62.7         4
2024                81.4  72.4  58.7        12
2025                81.5  74.2  58.2        12
2026                78.3  68.8   NaN         8

mejor cohorte al mes 6: 2023-11 (83.6%) · peor: 2023-09 (59.4%)


El triángulo de huecos de la derecha es la razón de ser de toda la página: **sólo las cohortes
antiguas han llegado a las edades altas**, así que la cola de la curva agregada la escriben ellas y
nadie más. Si además esas cohortes se comportaran distinto, la cola estaría sesgada.

Y se comportan distinto. La tabla por año de alta lo mide: las cohortes de 2026 retienen un **78,3%
al mes 3 y un 68,8% al mes 6**, frente al 81-84% y 72-74% de los años anteriores. Es decir, las altas
recientes retienen **peor**, y son justo las que todavía no han llegado a las edades altas. La cola
de la curva agregada, escrita por las cohortes de 2023 y 2024, es por tanto **optimista** para
proyectar lo que hará la base que entra hoy.

Por eso el modelo de cohortes de la página 2 se instancia con ponderación por recencia. Esta página
publica la matriz completa para que esa decisión se pueda auditar y no haya que creérsela.

## 7. LTV por canal de adquisición

Es el insumo de la página de cierre. Y tiene una trampa de supervivencia que hay que resolver antes
de comparar canales: **el LTV observado sólo se puede medir sobre suscripciones viejas**, y las
suscripciones viejas de un canal son las que ese canal consiguió retener.

In [12]:
CHANNELS = [c for c in base.acquisition_channel.dropna().unique()]
channel_curves = {ch: us.build_retention_curve(base[base.acquisition_channel == ch], min_cohort=10)
                  for ch in CHANNELS}

observed_by_channel = pd.DataFrame(
    {ch: c.values.reindex(range(0, 25)) for ch, c in channel_curves.items()}) * 100
print("Retención por canal, curva observada sin corregir la cola (%):")
print(observed_by_channel.loc[[3, 6, 12, 18, 24]].round(1).to_string())
print()
observed_spread = (observed_by_channel.loc[[3, 6, 12, 18]].max(axis=1)
                   - observed_by_channel.loc[[3, 6, 12, 18]].min(axis=1))
print("Dispersión entre canales (mejor menos peor), en puntos:")
print(observed_spread.round(1).to_string())

fig = go.Figure()
for ch, curve in channel_curves.items():
    fig.add_trace(go.Scatter(x=curve.values.index[:25], y=curve.values.to_numpy()[:25] * 100,
                             name=CHANNEL_LABEL[ch], mode="lines",
                             line=dict(color=CHANNEL_COLOR[ch], width=2.2)))
fig.update_layout(**{**PLOT_LAYOUT, "height": 430},
                  title="Retención por canal: los canales no se separan hasta el año",
                  xaxis_title="meses desde el alta", yaxis_title="% retenido")
fig.show()

Retención por canal, curva observada sin corregir la cola (%):
                    organic  paid_social  influencer_code  referral  direct_unknown  podcast_ads
months_since_start                                                                              
3                      80.4         82.1             77.9      80.9            81.5         81.1
6                      72.7         74.8             71.0      73.6            73.7         73.8
12                     58.9         64.6             52.5      56.8            60.4         54.4
18                     43.4         53.3             38.1      48.1            48.5         44.0
24                     34.1         47.7             31.1      38.6            36.8          NaN

Dispersión entre canales (mejor menos peor), en puntos:
months_since_start
3      4.3
6      3.8
12    12.1
18    15.2


### Antes de proyectar: la cola de la curva no se sostiene sola

Sumar la curva de retención hasta el mes 36 significa apoyarse en sus últimas edades, y ahí es donde
la estimación es más débil: a los 35 meses **la escribe una sola cohorte**. Conviene mirarlo antes
de multiplicar nada por ella.

In [13]:
tail_diag = pd.DataFrame({
    "retencion_pct": curve_eom.values * 100,
    "cohortes_detras": curve_eom.n_cohorts_by_age.reindex(curve_eom.values.index),
}).tail(14)
tail_diag["variacion_pp"] = tail_diag.retencion_pct.diff()
print("Cola de la curva agregada (sin regaladas):")
print(tail_diag.round(2).to_string())
print()
rises = tail_diag[tail_diag.variacion_pp > 0]
print(f"edades en las que la curva SUBE: {list(rises.index)} "
      f"-> imposible en una función de supervivencia")
print(f"tail_hazard que estima utils_subscriptions: {curve_eom.tail_hazard:+.4f}")
channel_tails = {ch: us.build_retention_curve(
                     base[base.acquisition_channel == ch], min_cohort=10).tail_hazard
                 for ch in CHANNELS}
print("tail_hazard por canal:",
      {k: round(v, 4) for k, v in sorted(channel_tails.items(), key=lambda kv: kv[1])})
print("-> negativo en", [k for k, v in channel_tails.items() if v < 0],
      ": la extrapolación haría crecer la retención con la edad")

Cola de la curva agregada (sin regaladas):
                    retencion_pct  cohortes_detras  variacion_pp
months_since_start                                              
22                          42.20               14           NaN
23                          39.15               13         -3.05
24                          39.24               12          0.09
25                          38.69               11         -0.55
26                          38.41               10         -0.28
27                          36.54                9         -1.87
28                          36.11                8         -0.43
29                          35.85                7         -0.27
30                          33.77                6         -2.07
31                          32.88                5         -0.90
32                          33.81                4          0.93
33                          35.81                3          2.00
34                          32.18              

tail_hazard por canal: {'organic': -0.0855, 'paid_social': -0.044, 'influencer_code': 0.0093, 'referral': 0.0406, 'podcast_ads': 0.0526, 'direct_unknown': 0.1016}
-> negativo en ['organic', 'paid_social'] : la extrapolación haría crecer la retención con la edad


Dos problemas, y los dos vienen de lo mismo. **La curva agregada deja de ser monótona** a partir del
mes 24: sube en tres edades. No es un error de cálculo sino el estimador enseñando su límite —divide
vivos entre altas de las cohortes que llegan a esa edad, y esa lista cambia de una edad a la
siguiente—. Pero una función de supervivencia no puede subir.

Y el `tail_hazard` que `utils_subscriptions` usa para extrapolar se estima justo sobre esas edades,
así que **sale negativo en dos canales**: extrapolar con él daría una retención que crece con la
edad. Para el forecast de la página 2 da igual, porque allí la curva se usa a horizontes de seis
meses; para un LTV a 36 meses, no.

La corrección es la mínima que hace falta: cortar la parte observada en la última edad con
suficientes cohortes detrás, imponer monotonía sobre ella —que es una propiedad de definición, no un
supuesto— y estimar el hazard de extrapolación sobre ese tramo fiable en vez de sobre el ruido final.

In [14]:
def reliable_curve(curve, horizon=LTV_HORIZON,
                   min_cohorts=TAIL_MIN_COHORTS, window=TAIL_WINDOW):
    """
    Curva utilizable para proyectar, con tres correcciones sobre `curve.at()`:

      1. se corta la parte observada en la última edad con `min_cohorts` cohortes detrás,
         porque más allá la escriben tres cohortes o menos;
      2. se impone monotonía con un mínimo acumulado: la retención es una función de
         supervivencia y no puede crecer, así que cualquier subida es ruido del estimador;
      3. el hazard de extrapolación se estima sobre las últimas `window` edades del tramo
         fiable, no sobre la cola ruidosa, y se acota a un mínimo positivo.

    Devuelve (serie 0..horizon-1, última edad fiable, hazard de la cola).
    """
    counts = curve.n_cohorts_by_age.reindex(curve.values.index).fillna(0)
    eligible = counts[counts >= min_cohorts]
    last = int(eligible.index.max()) if len(eligible) else int(curve.max_observed_age)
    observed = curve.values.loc[:last].cummin()
    hazards = (1 - observed / observed.shift(1)).dropna().tail(window)
    tail = float(min(max(hazards.mean(), 1e-3), 0.5))
    values = {int(a): float(observed.loc[a]) for a in observed.index}
    level = values[last]
    for age in range(last + 1, horizon):
        level *= (1 - tail)
        values[age] = level
    return pd.Series(values).sort_index(), last, tail

# ARPU por canal sobre los meses activos, y LTV proyectado = ARPU x suma de la curva.
arpu = (base[base.is_active_eom].groupby("acquisition_channel").recognized_revenue_eur.mean())
observed_ltv = (base.groupby(["acquisition_channel", "subscription_id"]).recognized_revenue_eur.sum()
                .groupby("acquisition_channel").mean())
mature = base[base.groupby("subscription_id").months_since_start.transform("max") >= 12]
observed_ltv_mature = (mature.groupby(["acquisition_channel", "subscription_id"])
                       .recognized_revenue_eur.sum().groupby("acquisition_channel").mean())

projected = {ch: reliable_curve(c) for ch, c in channel_curves.items()}

# La tabla de más arriba era la curva observada, útil para ver el problema de la cola. A partir
# de aquí todo —LTV y lo que se publica— usa la corregida, para que el JSON no lleve dos
# retenciones distintas del mismo canal.
retention_by_channel = pd.DataFrame(
    {ch: values.reindex(range(0, LTV_HORIZON)) * 100 for ch, (values, _, _) in projected.items()})
spread = (retention_by_channel.loc[[3, 6, 12, 18, 24]].max(axis=1)
          - retention_by_channel.loc[[3, 6, 12, 18, 24]].min(axis=1))
print("Retención corregida por canal (%):")
print(retention_by_channel.loc[[3, 6, 12, 18, 24]].round(1).to_string())
print()
print("Dispersión entre canales, curva corregida (puntos):")
print(spread.round(1).to_string())
print()

ltv_rows = []
for ch, curve in channel_curves.items():
    values, last_age, tail = projected[ch]
    months = float(values.sum())
    ltv_rows.append({
        "canal": ch,
        "suscripciones": int(base[base.acquisition_channel == ch].subscription_id.nunique()),
        "arpu_mes": float(arpu[ch]),
        "ultima_edad_fiable": int(last_age),
        "hazard_cola": float(tail),
        "meses_esperados": months,
        "ltv_proyectado": float(arpu[ch] * months),
        "ltv_observado": float(observed_ltv[ch]),
        "ltv_observado_maduras": float(observed_ltv_mature[ch]),
        "retencion_12m": float(values.loc[12] * 100),
        "retencion_24m": float(values.loc[24] * 100),
    })
ltv_by_channel = pd.DataFrame(ltv_rows).sort_values("ltv_proyectado", ascending=False)
print(f"LTV por canal (horizonte {LTV_HORIZON} meses, sólo la suscripción):")
print(ltv_by_channel.round(2).to_string(index=False))
print()
rank_proj = ltv_by_channel.set_index("canal").ltv_proyectado.rank(ascending=False)
rank_obs = ltv_by_channel.set_index("canal").ltv_observado_maduras.rank(ascending=False)
print("Puesto según LTV proyectado frente a LTV observado en suscripciones maduras:")
print(pd.DataFrame({"proyectado": rank_proj, "observado_maduras": rank_obs}).astype(int).to_string())

# Lo que cuesta el descuento de bienvenida en meses de vida esperados.
discount_months = {label: float(reliable_curve(c)[0].sum())
                   for label, c in (("con_descuento", disc), ("sin_descuento", plain))}
arpu_all = float(base[base.is_active_eom].recognized_revenue_eur.mean())
gap_months = discount_months["sin_descuento"] - discount_months["con_descuento"]
print()
print(f"meses de vida esperados: con descuento {discount_months['con_descuento']:.1f} · "
      f"sin descuento {discount_months['sin_descuento']:.1f}")
print(f"el descuento cuesta {gap_months:.1f} meses de vida y {gap_months * arpu_all:.0f} € de LTV "
      f"por alta captada")

Retención corregida por canal (%):
    organic  paid_social  influencer_code  referral  direct_unknown  podcast_ads
3      80.4         82.1             77.9      80.9            81.5         81.1
6      72.7         74.8             71.0      73.6            73.7         73.8
12     58.9         64.6             52.5      56.8            60.4         54.4
18     43.4         53.3             38.1      48.1            48.5         42.7
24     34.1         44.9             29.5      42.0            36.1         33.2

Dispersión entre canales, curva corregida (puntos):
3      4.3
6      3.8
12    12.1
18    15.2
24    15.4

LTV por canal (horizonte 36 meses, sólo la suscripción):
          canal  suscripciones  arpu_mes  ultima_edad_fiable  hazard_cola  meses_esperados  ltv_proyectado  ltv_observado  ltv_observado_maduras  retencion_12m  retencion_24m
    paid_social            990     25.53                  29         0.01            21.11          539.07         270.69                 


meses de vida esperados: con descuento 16.7 · sin descuento 21.8
el descuento cuesta 5.1 meses de vida y 128 € de LTV por alta captada


**Los canales no se distinguen hasta que pasa un año.** Al mes 3 la diferencia entre el mejor y el
peor es de 4 puntos —ruido, con estos tamaños—; al mes 12 son 12 puntos y al 18, **15 puntos**. Paid
social retiene un 53% a los 18 meses y el código de influencer un 38%. Cualquier evaluación de canal
hecha con tres meses de datos los habría declarado equivalentes.

(Esta tabla es la curva observada tal cual, que es la que hace falta para el diagnóstico de la
sección siguiente. Lo que se publica es la corregida, y sólo cambia donde tenía que cambiar: en el
podcast, cuyo tramo fiable acaba en el mes 16, la retención a 18 meses pasa del 44,0% al 42,7%.)

El orden cambia según cómo se mida, y ése es el aviso para la página 7: por **LTV observado sobre
suscripciones maduras** el podcast parece el mejor canal (573 €), porque en esa media sólo entran las
suscripciones que llegaron a los 12 meses —las que sobrevivieron—. Por **LTV proyectado**, que usa
toda la curva y por tanto también a las que cancelaron pronto, el podcast cae al cuarto puesto y paid
social pasa al primero: **539 € frente a 411 € del código de influencer**, un 31% de diferencia. El
LTV que hay que cruzar con el CAC es el proyectado.

Y hay un número que conviene llevarse a esa página: **el descuento de bienvenida cuesta 5,1 meses de
vida esperada** —16,7 frente a 21,8— es decir unos 128 € de LTV por alta captada. Ése es el lado del
coste que la criba del mes 2 no enseña.

Una advertencia sobre `direct_unknown`: **no es un canal**, es el cajón de las conversiones sin
touchpoint resuelto. Aparece en las tablas porque es el 29% de las altas y esconderlo falsearía los
totales, pero no se puede optimizar ni comprar más de él. La página 6 cuantifica ese hueco.

## 8. El comprador de máquina que se suscribe semanas después

La cuarta imperfección de cohortes: quien compra una máquina con cápsulas de regalo se suscribe
"varias semanas después, no inmediatamente — un join ingenuo por fecha exacta los perdería".

`int_machine_trial_conversions` resuelve el join con una ventana y deja además el contraste montado,
así que se puede medir exactamente cuánto cuesta el atajo.

In [15]:
trials = con.sql("select * from int_machine_trial_conversions").df()
trials["order_date"] = pd.to_datetime(trials.order_date)

matrix = pd.crosstab(trials.would_match_naive_same_day_join, trials.is_trial_conversion)
matrix.index = ["join ingenuo NO encuentra", "join ingenuo SÍ encuentra"]
matrix.columns = ["no es conversión", "es conversión"]
print("Compradores de máquina con trial de cápsulas:")
print(matrix.to_string())

real = int(trials.is_trial_conversion.sum())
naive_hits = int(trials.would_match_naive_same_day_join.sum())
tp = int((trials.would_match_naive_same_day_join & trials.is_trial_conversion).sum())
print()
print(f"conversiones reales (ventana resuelta): {real}")
print(f"filas que devuelve el join por fecha exacta: {naive_hits}, de las cuales aciertos: {tp}")
print(f"precisión del join ingenuo: {tp / naive_hits:.0%} · cobertura: {tp / real:.0%}")
print(f"de esas {naive_hits}, {int(trials[trials.would_match_naive_same_day_join].was_already_subscriber.sum())} "
      f"ya eran suscriptores antes de comprar la máquina")

lag = trials[trials.is_trial_conversion].conversion_lag_days
print()
print(f"días entre la compra y el alta: mediana {lag.median():.0f}, "
      f"p10 {lag.quantile(0.1):.0f}, p90 {lag.quantile(0.9):.0f}, mínimo {lag.min():.0f}")

fig = go.Figure(go.Histogram(x=lag, nbinsx=30, marker_color=C_BLUE))
fig.add_vline(x=0, line=dict(color=C_RED, width=2, dash="dash"),
              annotation_text="lo que encuentra un join por fecha exacta", annotation_position="top right")
fig.update_layout(**{**PLOT_LAYOUT, "height": 360, "hovermode": "closest"},
                  title="Días entre comprar la máquina y darse de alta",
                  xaxis_title="días", yaxis_title="conversiones")
fig.show()

Compradores de máquina con trial de cápsulas:
                           no es conversión  es conversión
join ingenuo NO encuentra               584            278
join ingenuo SÍ encuentra               113              0

conversiones reales (ventana resuelta): 278
filas que devuelve el join por fecha exacta: 113, de las cuales aciertos: 0
precisión del join ingenuo: 0% · cobertura: 0%
de esas 113, 113 ya eran suscriptores antes de comprar la máquina

días entre la compra y el alta: mediana 44, p10 16, p90 89, mínimo 1


In [16]:
# ¿Retienen distinto los convertidos desde máquina?
converted_ids = set(trials[trials.is_trial_conversion].subscription_id.dropna())
conv_panel = base[base.subscription_id.isin(converted_ids)]
rest_panel = base[~base.subscription_id.isin(converted_ids)]
conv_curve = us.build_retention_curve(conv_panel, min_cohort=5)
rest_curve = us.build_retention_curve(rest_panel, min_cohort=MIN_COHORT)
conv_compare = pd.DataFrame({"desde_maquina": conv_curve.values,
                             "resto": rest_curve.values}).loc[[3, 6, 12, 18]] * 100
conv_compare["dif_pp"] = conv_compare.desde_maquina - conv_compare.resto
print(f"Retención de los {len(converted_ids)} convertidos desde máquina frente al resto (%):")
print(conv_compare.round(1).to_string())

machine_first = con.sql("""
    select count(*) filter (where has_bundled_capsules_trial) trials,
           count(*) pedidos_maquina,
           round(avg(price_paid_eur), 2) precio_medio
    from fct_machine_orders
""").df()
print()
print(machine_first.to_string(index=False))

Retención de los 278 convertidos desde máquina frente al resto (%):
                    desde_maquina  resto  dif_pp
months_since_start                              
3                            85.7   80.8     4.9
6                            76.2   73.2     3.0
12                           66.3   59.0     7.3
18                           53.7   47.7     5.9

 trials  pedidos_maquina  precio_medio
   1188             3417        246.08


**El atajo no es que pierda conversiones: es que encuentra otra cosa.** El join por fecha exacta
devuelve 113 filas y **ninguna** de ellas es una conversión: las 113 son clientes que **ya eran
suscriptores** antes de comprar la máquina y que, por casualidad de calendario, tienen un alta el
mismo día. Precisión 0%, cobertura 0% sobre las 278 conversiones reales.

La razón está en el histograma: el alta llega con una mediana de **44 días** de retraso y nunca el
mismo día. Un join por fecha exacta no puede acertar ni una, por construcción.

Los convertidos desde máquina retienen **mejor** que el resto en todas las edades observadas, lo cual
tiene sentido de negocio —ya han hecho el desembolso grande y tienen el aparato en la cocina— y es
un argumento directo para la página 7: el coste de captar a un comprador de máquina no se juzga sólo
contra el margen de la máquina.

## 9. RFM sobre la compra puntual

Cambio de eje y de población. El RFM no mira suscripciones sino **clientes que compran en la tienda**
—online y boutique— con las tres variables clásicas: cuándo compró por última vez (*recency*), cuántas
veces (*frequency*) y cuánto (*monetary*).

Tiene una limitación de cobertura que hay que poner por delante, porque no se puede arreglar: las
compras de boutique sin fidelización no tienen cliente.

In [17]:
coverage = con.sql("""
    select channel,
           is_identified_customer                          as identificado,
           count(distinct order_id)                        as pedidos,
           round(sum(line_amount_eur))                     as eur
    from fct_shop_orders group by 1, 2 order by 1, 2
""").df()
coverage["pct_pedidos"] = coverage.pedidos / coverage.pedidos.sum() * 100
coverage["pct_eur"] = coverage.eur / coverage.eur.sum() * 100
print(coverage.round(1).to_string(index=False))
lost = coverage[~coverage.identificado]
print()
print(f"sin cliente: {lost.pedidos.sum():,} pedidos ({lost.pct_pedidos.sum():.1f}%) y "
      f"{lost.eur.sum():,.0f} € ({lost.pct_eur.sum():.1f}%) quedan fuera del RFM")

orders = con.sql("""
    select customer_key, order_id,
           max(order_date)      as order_date,
           max(order_total_eur) as order_total_eur
    from fct_shop_orders
    where customer_key is not null
    group by 1, 2
""").df()
orders["order_date"] = pd.to_datetime(orders.order_date)
print()
print(f"base del RFM: {orders.order_id.nunique():,} pedidos de "
      f"{orders.customer_key.nunique():,} clientes identificados")

channel  identificado  pedidos      eur  pct_pedidos  pct_eur
 online          True    22147 984448.0         55.5     56.0
  store         False    10924 472768.0         27.4     26.9
  store          True     6817 301258.0         17.1     17.1



sin cliente: 10,924 pedidos (27.4%) y 472,768 € (26.9%) quedan fuera del RFM

base del RFM: 28,964 pedidos de 9,368 clientes identificados


In [18]:
rfm = orders.groupby("customer_key").agg(
    ultima_compra=("order_date", "max"),
    frecuencia=("order_id", "nunique"),
    gasto=("order_total_eur", "sum")).reset_index()
rfm["recencia_dias"] = (AS_OF - rfm.ultima_compra).dt.days
print(rfm[["recencia_dias", "frecuencia", "gasto"]].describe().round(1).to_string())

def quintile(series, reverse=False):
    """
    Quintiles por rango medio, no por `qcut`.

    La frecuencia es discreta y está muy concentrada —uno de cada cuatro clientes tiene
    exactamente un pedido— así que los cortes de `qcut` caen dentro de un empate y fallan.
    El rango medio reparte los empates y mantiene el orden, a cambio de quintiles desiguales
    en la variable discreta, que es el precio honesto de tener sólo 14 valores distintos.
    """
    ranks = series.rank(method="average", pct=True)
    scores = np.ceil(ranks * RFM_QUANTILES).clip(1, RFM_QUANTILES).astype(int)
    return RFM_QUANTILES + 1 - scores if reverse else scores

rfm["R"] = quintile(rfm.recencia_dias, reverse=True)
rfm["F"] = quintile(rfm.frecuencia)
rfm["M"] = quintile(rfm.gasto)
rfm["rfm"] = rfm.R.astype(str) + rfm.F.astype(str) + rfm.M.astype(str)

print()
print("Clientes por puntuación (1 = peor, 5 = mejor):")
print(pd.DataFrame({k: rfm[k].value_counts().sort_index() for k in ("R", "F", "M")}).to_string())
print()
print("Cortes reales de cada quintil:")
cuts = rfm.groupby("R").recencia_dias.agg(["min", "max"]).rename(columns={"min": "rec_min", "max": "rec_max"})
cuts = cuts.join(rfm.groupby("F").frecuencia.agg(["min", "max"]).rename(
    columns={"min": "frec_min", "max": "frec_max"}))
cuts = cuts.join(rfm.groupby("M").gasto.agg(["min", "max"]).round(0).rename(
    columns={"min": "gasto_min", "max": "gasto_max"}))
print(cuts.to_string())

       recencia_dias  frecuencia   gasto
count         9368.0      9368.0  9368.0
mean           134.1         3.1   137.2
std            115.4         2.1   104.2
min              0.0         1.0     4.1
25%             50.0         1.0    58.8
50%            107.0         3.0   111.1
75%            187.0         4.0   187.8
max           1011.0        14.0   799.0



Clientes por puntuación (1 = peor, 5 = mejor):
      R     F     M
1  1872  2521  1873
2  1876  2105  1873
3  1854  1585  1874
4  1896  1116  1874
5  1870  2041  1874

Cortes reales de cada quintil:
   rec_min  rec_max  frec_min  frec_max  gasto_min  gasto_max
R                                                            
1      211     1011         1         1        4.0       49.0
2      133      210         2         2       49.0       89.0
3       84      132         3         3       89.0      136.0
4       39       83         4         4      136.0      212.0
5        0       38         5        14      212.0      799.0


In [19]:
def segment(row):
    r, f = row.R, row.F
    if r >= 4 and f >= 4: return "Campeones"
    if r >= 3 and f >= 3: return "Leales"
    if r >= 4 and f <= 2: return "Nuevos / prometedores"
    if r == 3 and f <= 2: return "Necesitan atención"
    if r <= 2 and f >= 4: return "En riesgo"
    if r <= 2 and f == 3: return "A punto de dormirse"
    if r == 1 and f <= 2: return "Perdidos"
    return "Hibernando"

rfm["segmento"] = rfm.apply(segment, axis=1)
segments = rfm.groupby("segmento").agg(
    clientes=("customer_key", "count"),
    recencia_mediana=("recencia_dias", "median"),
    frecuencia_media=("frecuencia", "mean"),
    gasto_medio=("gasto", "mean"),
    gasto_total=("gasto", "sum")).sort_values("gasto_total", ascending=False)
segments["pct_clientes"] = segments.clientes / segments.clientes.sum() * 100
segments["pct_ingreso"] = segments.gasto_total / segments.gasto_total.sum() * 100
print(segments.round(1).to_string())

fig = go.Figure()
fig.add_trace(go.Bar(x=segments.index, y=segments.pct_clientes, name="% de clientes",
                     marker_color=C_MUTED))
fig.add_trace(go.Bar(x=segments.index, y=segments.pct_ingreso, name="% del ingreso de tienda",
                     marker_color=C_BLUE))
fig.update_layout(**{**PLOT_LAYOUT, "height": 420, "barmode": "group", "hovermode": "x"},
                  title="Segmentos RFM: peso en clientes y en ingreso", yaxis_title="%")
fig.show()

champions = segments.loc["Campeones"]
at_risk = segments.loc["En riesgo"]
print()
print(f"Campeones: {champions.pct_clientes:.1f}% de los clientes y "
      f"{champions.pct_ingreso:.1f}% del ingreso")
print(f"En riesgo: {at_risk.pct_clientes:.1f}% de los clientes y "
      f"{at_risk.pct_ingreso:.1f}% del ingreso, con {at_risk.recencia_mediana:.0f} días sin comprar")

                       clientes  recencia_mediana  frecuencia_media  gasto_medio  gasto_total  pct_clientes  pct_ingreso
segmento                                                                                                                
Campeones                  1501              38.0               5.7        252.7     379360.0          16.0         29.5
Leales                     1602              94.0               4.1        181.8     291213.9          17.1         22.7
En riesgo                   960             193.0               5.2        233.2     223836.5          10.2         17.4
Nuevos / prometedores      1654              38.0               1.5         65.4     108249.1          17.7          8.4
A punto de dormirse         679             203.0               3.0        133.3      90519.2           7.2          7.0
Perdidos                   1179             291.0               1.4         64.7      76289.1          12.6          5.9
Hibernando                  930 


Campeones: 16.0% de los clientes y 29.5% del ingreso
En riesgo: 10.2% de los clientes y 17.4% del ingreso, con 193 días sin comprar


In [20]:
# El RFM se cruza con la suscripción y con el canal, que es lo que lo hace accionable.
enriched = rfm.merge(
    customers[["customer_key", "acquisition_channel", "is_subscriber",
               "has_active_subscription", "lifetime_revenue_eur", "shop_revenue_eur",
               "subscription_recognized_eur", "machine_revenue_eur", "is_machine_buyer"]],
    on="customer_key", how="left")

rfm_matrix = (rfm.pivot_table(index="R", columns="F", values="customer_key", aggfunc="count")
              .reindex(index=range(1, 6), columns=range(1, 6)))
rfm_monetary = (rfm.pivot_table(index="R", columns="F", values="gasto", aggfunc="mean")
                .reindex(index=range(1, 6), columns=range(1, 6)))
print("Clientes por casilla R x F:")
print(rfm_matrix.fillna(0).astype(int).to_string())

by_segment = enriched.groupby("segmento").agg(
    clientes=("customer_key", "count"),
    pct_suscriptor=("is_subscriber", lambda s: s.mean() * 100),
    pct_suscriptor_activo=("has_active_subscription", lambda s: s.mean() * 100),
    pct_comprador_maquina=("is_machine_buyer", lambda s: s.mean() * 100),
    ltv_medio=("lifetime_revenue_eur", "mean")).sort_values("ltv_medio", ascending=False)
print("Segmento RFM cruzado con la relación de suscripción:")
print(by_segment.round(1).to_string())
print()

customer_ltv = customers.groupby("acquisition_channel").agg(
    clientes=("customer_key", "count"),
    pct_suscriptor=("is_subscriber", lambda s: s.mean() * 100),
    ltv_cliente=("lifetime_revenue_eur", "mean"),
    de_suscripcion=("subscription_recognized_eur", "mean"),
    de_tienda=("shop_revenue_eur", "mean"),
    de_maquina=("machine_revenue_eur", "mean")).sort_values("ltv_cliente", ascending=False)
print("LTV de cliente por canal de adquisición (suscripción + tienda + máquina):")
print(customer_ltv.round(1).to_string())
print()
print(f"dispersión del LTV de cliente entre canales: "
      f"{(customer_ltv.ltv_cliente.max() / customer_ltv.ltv_cliente.min() - 1) * 100:.1f}%")

Clientes por casilla R x F:
F    1    2    3    4    5
R                         
1  694  485  313  191  189
2  502  428  366  205  375
3  455  408  295  240  456
4  447  375  336  247  491
5  423  409  275  233  530
Segmento RFM cruzado con la relación de suscripción:
                       clientes  pct_suscriptor  pct_suscriptor_activo  pct_comprador_maquina  ltv_medio
segmento                                                                                                
En riesgo                   960            45.9                   14.8                   28.6      500.6
Campeones                  1501            45.4                   15.5                   27.5      496.9
Leales                     1602            43.4                   16.7                   26.7      402.6
A punto de dormirse         679            43.9                   17.7                   27.4      367.1
Perdidos                   1179            43.6                   18.0                   25.1      2

**La forma del RFM es la esperada y el detalle es donde está el negocio.** Los *Campeones* son el 16%
de los clientes y el 29,5% del ingreso de tienda; los *En riesgo* son el 10,2% de los clientes pero
el 17,4% del ingreso, con una mediana de 193 días sin comprar. Ése es el segmento con retorno: son
clientes de gasto alto (233 € de media, por encima incluso de la media de los Leales) que llevan
medio año sin aparecer.

El cruce con la suscripción deja una lectura que cambia la prioridad: **la proporción de suscriptores
es prácticamente la misma en todos los segmentos RFM** (39-46%). Ser buen cliente de tienda y ser
suscriptor son dos relaciones **independientes**, no dos escalones de la misma escalera. No se puede
usar el RFM de tienda como proxy del valor de suscripción ni al revés, y la página 7 tiene que sumar
las dos patas por separado.

Y el aviso para esa página: **el LTV de cliente apenas se mueve entre canales** —de 278 € a 289 €, un
3,8% de dispersión— mientras que el LTV de *suscripción proyectado* de la sección 7 iba de 411 € a
539 €, un 31%. La diferencia es que el LTV de cliente mezcla tienda y máquina, que no dependen del canal de
captación de la suscripción, y diluye la señal. **Para cruzar con el CAC hay que usar el LTV
proyectado de suscripción**, no el agregado por cliente.

Por último, la cobertura: el RFM sólo ve a los 9.368 clientes identificados. Los 10.924 pedidos de
boutique sin fidelización —el **27,4% de los pedidos y el 26,9% del ingreso** de tienda— no se pueden
puntuar, y no es un problema que se pueda arreglar aguas arriba: esa venta ocurrió sin que nadie
diera un identificador. Cualquier cifra de "ingreso por cliente" de esta página es, por construcción,
ingreso por cliente **identificado**.

## 10. Volcado a `analysis/outputs/cohorts_rfm.json`

In [21]:
def curve_records(curve):
    # Curva publicada: tramo fiable observado mas extrapolacion, ya monotona.
    values, last_age, tail = reliable_curve(curve)
    return [{"age": int(a), "retention": float(v),
             "n_cohorts": int(curve.n_cohorts_by_age.get(a, 0)),
             "is_extrapolated": bool(a > last_age)}
            for a, v in values.items()]

payload = {
    "meta": {
        "page": "05_cohortes_rfm",
        "title": "Cohortes y RFM",
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_tables": ["fct_subscriptions_monthly", "dim_subscriptions", "dim_customers",
                          "fct_shop_orders", "int_machine_trial_conversions"],
        "as_of": AS_OF.strftime("%Y-%m-%d"),
        "cohort_range": [subs.cohort_month.min().strftime("%Y-%m-%d"),
                         subs.cohort_month.max().strftime("%Y-%m-%d")],
        "n_subscriptions": int(len(subs)),
        "n_customers": int(len(customers)),
        "min_cohort_size": MIN_COHORT,
        "ltv_horizon_months": LTV_HORIZON,
        "tail_note": ("La cola de la curva agregada la escriben 1-4 cohortes y deja de ser "
                      "monótona a partir del mes 24; el tail_hazard de utils_subscriptions "
                      "sale negativo en dos canales. Las curvas publicadas cortan el tramo "
                      f"observado en la última edad con {TAIL_MIN_COHORTS} cohortes detrás, "
                      "imponen monotonía y extrapolan con el hazard de ese tramo fiable."),
        "active_definition": "is_active_eom",
        "active_definition_note": (
            "La retención se mide sobre is_active_eom, que ignora las pausas. Con la definición "
            "neta de pausas la curva sale 2-4 puntos por debajo en todas las edades porque la "
            "pausa estacional de agosto entra como churn a una edad distinta en cada cohorte."),
        "right_censored_pct": float(subs.is_right_censored.mean() * 100),
    },
    "population": {
        "note": ("Tres grupos disjuntos: ninguna suscripción regalada lleva descuento de "
                 "bienvenida."),
        "groups": json.loads(population.reset_index().to_json(orient="records")),
    },
    "retention": {
        "base": curve_records(curve_eom),
        "base_net_of_pauses": curve_records(curve_net),
        "gifted": curve_records(gift_curve),
        "with_discount": curve_records(disc),
        "without_discount": curve_records(plain),
        "machine_converts": curve_records(conv_curve),
        "pause_rate_by_calendar_month": [
            {"month": int(m), "pct_paused": float(v)} for m, v in pause_by_month.items()],
    },
    "gifted": {
        "imperfection": "Suscripciones regaladas",
        "n": int(subs.is_gifted.sum()),
        "treatment": ("Se excluyen de la curva principal y se publican aparte: su caída es el fin "
                      "del término del regalo, no una decisión de consumo."),
        "cliff_months_3_to_6_pp": float(cliff),
        "cancel_reason_mix": json.loads(reasons.reset_index().to_json(orient="records")),
        "cancel_reason_note": ("No existe un motivo de cancelación específico de regalo: se "
                               "reparten entre los mismos cinco motivos que el resto, así que "
                               "filtrar por cancel_reason no los encuentra."),
    },
    "welcome_discount": {
        "imperfection": "Descuento de bienvenida y pico de cancelación al acabarse",
        "n_with_discount": int((~subs.is_gifted & subs.had_welcome_discount).sum()),
        "share_pct": float(ng_subs.had_welcome_discount.mean() * 100),
        "assignment_balance": json.loads(balance.to_json(orient="records")),
        "assignment_note": ("El descuento está repartido al azar: ningún contraste chi-cuadrado "
                            "rechaza la independencia frente a canal, tier, plan o año de alta. "
                            "Eso es lo que autoriza a leer la diferencia de curvas como efecto."),
        "retention": json.loads(retention_by_discount.reset_index().to_json(orient="records")),
        "hazard": json.loads(hazard_by_discount.reset_index().to_json(orient="records")),
        "excess_cancellations": [
            {"age": int(a), "excess": float(v)} for a, v in excess.head(12).items()],
        "excess_total_months_1_2": float(excess.loc[spike_ages].sum()),
        "excess_pct_of_discounted_cohort": float(
            excess.loc[spike_ages].sum() / exposure[True].loc[0] * 100),
        "revenue_by_age": json.loads(revenue_age.reset_index().to_json(orient="records")),
        "revenue_note": ("El descuento NO se ve en el MRR contratado (29,87 € contra 29,97 € en el "
                         "mes 0) y sí en el ingreso reconocido, que en el mes 0 es la mitad. Quien "
                         "compare cohortes por MRR no lo verá."),
    },
    "plan_changes": {
        "imperfection": "Downgrades de plan",
        "decision": ("Cuentan como 'sigue activo' en la curva de retención. En estos datos un "
                     "downgrade acorta el ciclo de compromiso (annual->quarterly->monthly), lo que "
                     "SUBE el MRR contratado un 6,3% porque desaparece el descuento por "
                     "compromiso, y no adelanta la cancelación."),
        "open_question": ("docs/data_imperfections.md y el comentario del generador describen el "
                          "downgrade como 'menos ingreso', pero el código mueve el plan hacia el "
                          "ciclo corto, que es el más caro por mes. Pendiente de revisar con "
                          "Diego: o se pretendía mover el tier, o el catálogo debería decir que "
                          "lo que se reduce es el compromiso."),
        "transitions": transitions,
        "n_downgrade_subscriptions": int(panel[panel.is_downgrade_month].subscription_id.nunique()),
        "n_upgrade_subscriptions": int(panel[panel.is_upgrade_month].subscription_id.nunique()),
        "hazard_after_downgrade_pct": float(h_after * 100),
        "hazard_control_pct": float(h_control * 100),
        "logo_vs_revenue_retention": json.loads(
            logo_vs_revenue.reset_index().to_json(orient="records")),
        "plan_mix_by_age": json.loads(plan_mix.head(25).reset_index().to_json(orient="records")),
        "gap_decomposition": {
            "note": ("Tres curvas sobre la misma base: logo (altas que no han cancelado), MRR de "
                     "tarifa (incluye el de las pausas) y MRR cobrable (el que entra ese mes). "
                     "logo - tarifa aísla el mix; tarifa - cobrable aísla la pausa."),
            "ages_averaged": "3-18",
            "efecto_mix_pp": float(effects.efecto_mix_pp),
            "efecto_pausa_pp": float(effects.efecto_pausa_pp),
            "gap_total_pp": float(effects.gap_total_pp),
            "pause_share_of_gap_pct": float(pause_share),
            "verdict": ("El hueco de 2-4 puntos NO lo abren los downgrades ni el mix de plan "
                        "(-0,06 puntos de media, pese a que el mix sí se desplaza cinco puntos): "
                        "lo abre la pausa, que explica el 102% del total."),
        },
    },
    "paused_revenue": {
        "imperfection": ("Pausas estacionales: retiran ingreso sin retirar cliente. Es el evento "
                         "que obliga a separar retención de logo de retención de ingreso."),
        "mrr_withdrawn_eur": float(paused_total),
        "pct_of_gross_mrr": float(paused_total / gross_total * 100),
        "subscriptions_affected": int(base[base.is_paused].subscription_id.nunique()),
        "subscription_months_paused": int(base.is_paused.sum()),
        "decision": ("En la curva de logo la pausa cuenta como activa, porque no es abandono y "
                     "vuelve. En la de ingreso no puede contar, porque ese mes no entra un euro. "
                     "Se publican las dos: la distancia entre ellas es la métrica."),
        "pause_rate_by_age": [
            {"age": int(a), "pct_paused": float(v)}
            for a, v in (base[base.is_active_eom].groupby("months_since_start")
                         .is_paused.mean().head(25) * 100).items()],
    },
    "cohort_matrix": {
        "note": ("Una fila por cohorte de alta con al menos "
                 f"{MIN_COHORT} altas; celdas vacías = la cohorte no ha llegado a esa edad."),
        "cohorts": [d.strftime("%Y-%m-%d") for d in cohort_matrix.index],
        "sizes": [int(v) for v in cohort_sizes.to_numpy()],
        "retention_pct": [
            {"cohort": d.strftime("%Y-%m-%d"),
             **{str(age): (None if pd.isna(v) else float(v))
                for age, v in cohort_matrix.loc[d].items()}}
            for d in cohort_matrix.index],
        "by_cohort_year": json.loads(by_year.reset_index().to_json(orient="records")),
        "survivorship_note": ("Las cohortes de 2026 retienen un 78,3% al mes 3 y un 68,8% al mes 6 "
                              "frente al 81-84% y 72-74% de los años anteriores, y son las que "
                              "todavía no han llegado a las edades altas: la cola de la curva "
                              "agregada es optimista."),
    },
    "ltv_by_channel": {
        "note": ("LTV de suscripción proyectado = ARPU mensual del canal x suma de su curva de "
                 f"retención a {LTV_HORIZON} meses. Es el que hay que cruzar con el CAC."),
        "warning": ("ltv_observado_maduras sólo promedia suscripciones que han llegado a los 12 "
                    "meses, es decir las que el canal consiguió retener: ordena los canales al "
                    "revés que el proyectado y es una trampa de supervivencia."),
        "direct_unknown_note": ("direct_unknown no es un canal sino el cajón de las conversiones "
                                "sin touchpoint resuelto. La página 6 cuantifica ese hueco."),
        "channels": json.loads(ltv_by_channel.to_json(orient="records")),
        "retention_by_channel": [
            {"age": int(a), **{ch: (None if pd.isna(retention_by_channel.loc[a, ch])
                                    else float(retention_by_channel.loc[a, ch]))
                               for ch in retention_by_channel.columns}}
            for a in retention_by_channel.index],
        "spread_pp": {str(a): float(v) for a, v in spread.items()},
        "customer_ltv_by_channel": json.loads(customer_ltv.reset_index().to_json(orient="records")),
        "customer_ltv_note": ("El LTV de cliente (suscripción + tienda + máquina) apenas varía "
                              "entre canales porque tienda y máquina no dependen del canal de "
                              "captación. Para el cruce con CAC hay que usar el de suscripción."),
    },
    "machine_conversion": {
        "imperfection": "Comprador de máquina con trial que se suscribe semanas después",
        "n_trials": int(len(trials)),
        "n_conversions": int(real),
        "naive_same_day_rows": int(naive_hits),
        "naive_true_positives": int(tp),
        "naive_precision_pct": float(tp / naive_hits * 100),
        "naive_recall_pct": float(tp / real * 100),
        "lag_days": {"median": float(lag.median()), "p10": float(lag.quantile(0.1)),
                     "p90": float(lag.quantile(0.9)), "min": float(lag.min())},
        "note": ("El join por fecha exacta devuelve 113 filas y ninguna es una conversión: son "
                 "clientes que ya eran suscriptores antes de comprar la máquina."),
        "retention_vs_rest": json.loads(conv_compare.reset_index().to_json(orient="records")),
    },
    "rfm": {
        "population": "clientes con al menos un pedido de tienda identificado",
        "as_of": AS_OF.strftime("%Y-%m-%d"),
        "n_customers": int(len(rfm)),
        "n_orders": int(orders.order_id.nunique()),
        "coverage": json.loads(coverage.to_json(orient="records")),
        "coverage_note": (f"{int(lost.pedidos.sum())} pedidos de boutique sin fidelización "
                          f"({lost.pct_pedidos.sum():.1f}% de los pedidos, "
                          f"{lost.pct_eur.sum():.1f}% del ingreso de tienda) no se pueden puntuar."),
        "scoring_note": ("Quintiles por rango medio y no por qcut: la frecuencia es discreta y uno "
                         "de cada cuatro clientes tiene exactamente un pedido, así que los cortes "
                         "de qcut caen dentro de un empate."),
        "score_distribution": {
            k: {str(score): int(n) for score, n in rfm[k].value_counts().sort_index().items()}
            for k in ("R", "F", "M")},
        "quintile_cuts": json.loads(cuts.reset_index().to_json(orient="records")),
        "segments": json.loads(segments.reset_index().to_json(orient="records")),
        "segments_vs_subscription": json.loads(by_segment.reset_index().to_json(orient="records")),
        "rf_matrix": [
            {"R": int(r), **{f"F{int(f)}": (None if pd.isna(rfm_matrix.loc[r, f])
                                            else int(rfm_matrix.loc[r, f]))
                             for f in rfm_matrix.columns}}
            for r in rfm_matrix.index],
        "rf_matrix_avg_spend": [
            {"R": int(r), **{f"F{int(f)}": (None if pd.isna(rfm_monetary.loc[r, f])
                                            else round(float(rfm_monetary.loc[r, f]), 2))
                             for f in rfm_monetary.columns}}
            for r in rfm_monetary.index],
        "independence_note": ("La proporción de suscriptores es prácticamente la misma en todos "
                             "los segmentos RFM (39-46%): ser buen cliente de tienda y ser "
                             "suscriptor son relaciones independientes."),
        "customers_note": ("El detalle por cliente no se publica: son 9.368 filas que el informe "
                           "no usa y que multiplicarían por diez el tamaño del JSON. Lo que el "
                           "informe necesita son los segmentos y la matriz R x F."),
    },
    "insights": [
        ("El 56% de las suscripciones sigue viva al cierre del histórico, así que cualquier métrica "
         "de duración media está sesgada a la baja. Todo se construye como curva de retención por "
         "edad, que usa la información de las vivas hasta donde llega."),
        ("La retención se mide sobre is_active_eom y no sobre los activos netos de pausas: la pausa "
         "es un fenómeno de mes natural (techo en agosto, 13,5%) y entraría en la curva como churn "
         "a una edad distinta en cada cohorte, restando 2-4 puntos en todas las edades."),
        ("Las suscripciones regaladas no tienen curva sino acantilado: pierden 65 puntos entre el "
         "mes 3 y el 6 al acabarse el término. Y no hay un motivo de cancelación que las delate, "
         "así que se aíslan por is_gifted, no por cancel_reason."),
        ("El descuento de bienvenida está repartido al azar (ningún chi-cuadrado lo rechaza), lo "
         "que autoriza a leer la diferencia de curvas como efecto: +7,1 puntos de hazard en el mes "
         "1 y +14,2 en el mes 2, unas 520 cancelaciones de más, el 19,4% de las altas con "
         "descuento."),
        ("A partir del mes 3 los dos grupos se comportan igual: el descuento no deteriora la "
         "retención a largo plazo, produce una criba única. La pregunta de negocio no es si daña "
         "la retención sino si el 81% que sobrevive paga el descuento y la captación del resto."),
        ("El descuento es invisible en el MRR contratado (29,87 € contra 29,97 € en el mes 0) y "
         "evidente en el ingreso reconocido, que en el mes 0 es la mitad. La medida que se elija "
         "decide si la imperfección se ve o no."),
        ("Un downgrade en estos datos no baja el precio: acorta el compromiso de 6,7 a 1,8 meses y "
         "SUBE el MRR un 6,3%, porque desaparece el descuento por ciclo largo. Tampoco adelanta la "
         "cancelación (3,83% frente a 3,89% mensual). Cuentan como activos, sin asterisco."),
        ("El hueco de 2-4 puntos entre retención de logo y de ingreso no lo abren los downgrades "
         "ni el mix de plan (-0,06 puntos de media, aunque el mix sí se desplaza cinco puntos): lo "
         "abre la pausa, que explica el 102% del hueco. El gráfico que enseña que el mix se mueve "
         "no demuestra que el mix importe."),
        ("La pausa es el único evento que retira ingreso sin retirar cliente: 55.360 € de MRR "
         "aparcado, el 3,9% del de tarifa, sobre 1.064 suscripciones. En la curva de logo cuenta "
         "como activa y en la de ingreso no puede contar; la distancia entre las dos curvas es "
         "precisamente la métrica que hay que publicar."),
        ("Los canales de adquisición no se distinguen hasta que pasa un año: 4 puntos de "
         "dispersión al mes 3 y 15 al mes 18, con paid social al 53% y código de influencer al "
         "38%. Evaluar un canal con tres meses de datos los declara equivalentes."),
        ("El LTV observado sobre suscripciones maduras ordena los canales al revés que el "
         "proyectado, porque sólo promedia las que sobrevivieron: el podcast pasa del primer "
         "puesto al cuarto. El que hay que cruzar con el CAC es el proyectado, de 411 € "
         "(código de influencer) a 539 € (paid social)."),
        ("El descuento de bienvenida cuesta 5,1 meses de vida esperada (16,7 frente a 21,8) y unos "
         "128 € de LTV por alta captada. Es el lado del coste que la criba del mes 2 no enseña."),
        ("La cola de la curva agregada la escriben 1-4 cohortes y deja de ser monótona a partir "
         "del mes 24; el tail_hazard de utils_subscriptions sale negativo en dos canales, lo que "
         "haría crecer la retención con la edad. Para un LTV a 36 meses hay que cortar el tramo "
         "observado donde deja de sostenerse e imponer monotonía."),
        ("El join por fecha exacta para encontrar conversiones de comprador de máquina devuelve "
         "113 filas con precisión 0% y cobertura 0%: las 113 ya eran suscriptores. Las 278 "
         "conversiones reales llegan con una mediana de 44 días de retraso."),
        ("Ser buen cliente de tienda y ser suscriptor son independientes: la proporción de "
         "suscriptores es 39-46% en todos los segmentos RFM. El valor de las dos patas hay que "
         "sumarlo, no deducirlo de una."),
        ("El RFM no puede puntuar el 27,4% de los pedidos de tienda ni el 26,9% de su ingreso, "
         "porque la venta de boutique sin fidelización no tiene cliente. No es un fallo del "
         "pipeline: esa compra ocurrió sin identificador."),
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)
print(f"Guardado {OUTPUT_PATH.relative_to(PROJECT_ROOT)} "
      f"({OUTPUT_PATH.stat().st_size / 1024:.0f} KB)")
print("Claves de primer nivel:", list(payload))

Guardado analysis\outputs\cohorts_rfm.json (124 KB)
Claves de primer nivel: ['meta', 'population', 'retention', 'gifted', 'welcome_discount', 'plan_changes', 'paused_revenue', 'cohort_matrix', 'ltv_by_channel', 'machine_conversion', 'rfm', 'insights']


In [22]:
with open(OUTPUT_PATH, encoding="utf-8") as handle:
    reloaded = json.load(handle)

checks = {
    "tres poblaciones": len(reloaded["population"]["groups"]) == 3,
    "curva base": len(reloaded["retention"]["base"]) >= 30,
    "curva de regaladas": len(reloaded["retention"]["gifted"]) > 0,
    "curvas por descuento": (len(reloaded["retention"]["with_discount"]) > 0
                             and len(reloaded["retention"]["without_discount"]) > 0),
    "retención monótona en todas las curvas publicadas": all(
        a["retention"] >= b["retention"] - 1e-9
        for serie in reloaded["retention"].values()
        if isinstance(serie, list) and serie and "retention" in serie[0]
        for a, b in zip(serie, serie[1:])),
    "curvas proyectadas al horizonte del LTV": all(
        len(reloaded["retention"][k]) == LTV_HORIZON
        for k in ("base", "with_discount", "without_discount")),
    "pausa por mes natural": len(reloaded["retention"]["pause_rate_by_calendar_month"]) == 12,
    "balance del descuento": len(reloaded["welcome_discount"]["assignment_balance"]) == 4,
    "descuento repartido al azar": all(
        r["p_valor"] > ALPHA for r in reloaded["welcome_discount"]["assignment_balance"]),
    "exceso concentrado en los meses 1-2": (
        reloaded["welcome_discount"]["excess_total_months_1_2"] > 400),
    "transiciones de plan": len(reloaded["plan_changes"]["transitions"]) >= 4,
    "la pausa explica el hueco, no el mix": (
        abs(reloaded["plan_changes"]["gap_decomposition"]["efecto_mix_pp"]) < 0.5
        and reloaded["plan_changes"]["gap_decomposition"]["efecto_pausa_pp"] > 2),
    "ingreso pausado cuadra con el panel": (
        abs(reloaded["paused_revenue"]["pct_of_gross_mrr"] - 3.87) < 0.1),
    "matriz de cohortes": (len(reloaded["cohort_matrix"]["retention_pct"])
                           == len(reloaded["cohort_matrix"]["cohorts"])
                           == len(reloaded["cohort_matrix"]["sizes"])),
    "seis canales con LTV": len(reloaded["ltv_by_channel"]["channels"]) == 6,
    "LTV proyectado > observado": all(
        c["ltv_proyectado"] > c["ltv_observado"] for c in reloaded["ltv_by_channel"]["channels"]),
    "conversión de máquina": (reloaded["machine_conversion"]["naive_true_positives"] == 0
                              and reloaded["machine_conversion"]["n_conversions"] > 0),
    "cobertura del RFM": len(reloaded["rfm"]["coverage"]) == 3,
    "ocho segmentos RFM": len(reloaded["rfm"]["segments"]) == 8,
    "matriz R x F completa": (len(reloaded["rfm"]["rf_matrix"]) == 5
                              and len(reloaded["rfm"]["rf_matrix_avg_spend"]) == 5),
    "la matriz R x F suma el total de clientes": (
        sum(v for row in reloaded["rfm"]["rf_matrix"]
            for k, v in row.items() if k != "R" and v) == len(rfm)),
    "segmentos suman el total": (
        sum(s["clientes"] for s in reloaded["rfm"]["segments"]) == len(rfm)),
}
for label, ok in checks.items():
    print(f"  {'OK ' if ok else 'FALLO'} {label}")
assert all(checks.values()), "El JSON de salida no tiene la forma esperada."
print()
print("JSON verificado.")

  OK  tres poblaciones
  OK  curva base
  OK  curva de regaladas
  OK  curvas por descuento
  OK  retención monótona en todas las curvas publicadas
  OK  curvas proyectadas al horizonte del LTV
  OK  pausa por mes natural
  OK  balance del descuento
  OK  descuento repartido al azar
  OK  exceso concentrado en los meses 1-2
  OK  transiciones de plan
  OK  la pausa explica el hueco, no el mix
  OK  ingreso pausado cuadra con el panel
  OK  matriz de cohortes
  OK  seis canales con LTV
  OK  LTV proyectado > observado
  OK  conversión de máquina
  OK  cobertura del RFM
  OK  ocho segmentos RFM
  OK  matriz R x F completa
  OK  la matriz R x F suma el total de clientes
  OK  segmentos suman el total

JSON verificado.


## Conclusiones

1. **La censura por la derecha manda sobre el método.** El 56% de las suscripciones sigue viva, así
   que "cuánto dura una suscripción" no se puede calcular como media: toda la página se construye
   sobre curvas de retención por edad, que usan lo que se sabe de las vivas sin inventarse el resto.
2. **Elegir la definición de activo es elegir qué historia se cuenta.** Medir la retención sobre los
   activos netos de pausas mete la estacionalidad de agosto dentro de la curva de edad y resta 2-4
   puntos en todas las edades. La pausa es calendario y se trata como calendario.
3. **Las regaladas no son una cohorte, y el campo que parecía servir para aislarlas no sirve.** Su
   curva es un acantilado de 65 puntos entre el mes 3 y el 6, y su motivo de cancelación se recicla
   del catálogo normal. Se aíslan por `is_gifted` y se publican aparte.
4. **El descuento de bienvenida es una criba, no un deterioro.** Está repartido al azar —comprobado
   antes de restar nada—, cuesta unas 520 cancelaciones concentradas en los meses 1 y 2, y a partir
   del mes 3 los dos grupos son indistinguibles. Y sólo se ve en el ingreso reconocido: en MRR
   contratado el mes 0 de las dos cohortes es idéntico.
5. **La pregunta abierta del catálogo tenía respuesta, pero con otro protagonista.** Los downgrades
   no reducen el ingreso: acortan el compromiso y suben el MRR un 6,3%. Cuentan como activos. El
   evento que de verdad retira ingreso sin retirar cliente es la **pausa**, y explica el 102% del
   hueco entre retención de logo y de ingreso. El mix de plan, que era la explicación intuitiva y
   tenía un gráfico a favor, aporta −0,06 puntos: se mueve sin mover el dinero. Que algo se
   desplace no demuestra que pese; hay que descomponer y medir.
6. **Un canal no se juzga a tres meses.** La dispersión entre canales es de 4 puntos al mes 3 y de
   15 al mes 18. Y el LTV que hay que llevar a la página de cierre es el **proyectado** con la curva
   de cada canal —de 411 € a 539 €—, no el observado sobre suscripciones maduras, que ordena los
   canales al revés porque sólo mide a los supervivientes. Esa curva, además, hay que cortarla donde
   deja de estar sostenida por cohortes suficientes: su cola la escriben tres cohortes y no es ni
   siquiera monótona.
7. **Dos relaciones independientes, no una escalera.** El RFM de tienda y la suscripción no se
   predicen mutuamente: la proporción de suscriptores es la misma en Campeones que en Perdidos. El
   valor de un cliente es la suma de dos patas que hay que medir por separado, y eso es exactamente
   lo que la página 7 tiene que cruzar con el CAC.